## SEINE Video Generation model

In [1]:
import os
import sys
import math
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'SEINE'))

import utils
from diffusion import create_diffusion

import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
import argparse
import torchvision

from einops import rearrange
from models import get_models
from torchvision.utils import save_image
from diffusers.models import AutoencoderKL
from models.clip import TextEmbedder
from omegaconf import OmegaConf
from PIL import Image
import numpy as np
from torchvision import transforms
from SEINE import video_transforms
# from dataset import video_transforms
from utils import mask_generation_before
from natsort import natsorted
from diffusers.utils.import_utils import is_xformers_available
import pdb
import datetime

class SeineModel:
    def __init__(self, args):
        print('Initializing SEINE model...')

        if args.seed:
            torch.manual_seed(args.seed)
        torch.set_grad_enabled(False)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        if args.ckpt is None:
            raise ValueError("Please specify a checkpoint path using --ckpt <path>")

        # load model
        self.latent_h = args.image_size[0] // 8
        self.latent_w = args.image_size[1] // 8
        self.image_h = args.image_size[0]
        self.image_w = args.image_size[1]
        self.model = get_models(args).to(self.device)

        if args.enable_xformers_memory_efficient_attention:
            if is_xformers_available():
                self.model.enable_xformers_memory_efficient_attention()
            else:
                raise ValueError("xformers is not available. Make sure it is installed correctly")

        ckpt_path = args.ckpt 
        state_dict = torch.load(ckpt_path, map_location=lambda storage, loc: storage)['ema']
        self.model.load_state_dict(state_dict)

        self.model.eval()
        pretrained_model_path = args.pretrained_model_path
        self.diffusion = create_diffusion(str(args.num_sampling_steps))
        self.vae = AutoencoderKL.from_pretrained(pretrained_model_path, subfolder="vae").to(self.device)
        self.text_encoder = TextEmbedder(pretrained_model_path).to(self.device)
        if args.use_fp16:
            # print('Warning: using half percision for inferencing!')
            self.vae.to(dtype=torch.float16)
            self.model.to(dtype=torch.float16)
            self.text_encoder.to(dtype=torch.float16)

        self.mask_type = args.mask_type
        self.num_frames = args.num_frames
        self.use_fp16 = args.use_fp16
        self.do_classifier_free_guidance = args.do_classifier_free_guidance
        self.sample_method = args.sample_method
        self.cfg_scale = args.cfg_scale
        self.use_mask = args.use_mask

        print('Initialization complete!')

    def get_input(self, input_path):
        transform_video = transforms.Compose([
                            video_transforms.ToTensorVideo(), # TCHW
                            video_transforms.ResizeVideo((self.image_h, self.image_w)),
                            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5], inplace=True)
                        ])
        if input_path is not None:
            print(f'Loading video from {input_path}...')
            if os.path.isdir(input_path):
                file_list = os.listdir(input_path)
                video_frames = []
                if self.mask_type.startswith('onelast'):
                    num = int(self.mask_type.split('onelast')[-1])
                    # get first and last frame
                    first_frame_path = os.path.join(input_path, natsorted(file_list)[0])
                    last_frame_path = os.path.join(input_path, natsorted(file_list)[-1])
                    first_frame = torch.as_tensor(np.array(Image.open(first_frame_path), dtype=np.uint8, copy=True)).unsqueeze(0)
                    last_frame = torch.as_tensor(np.array(Image.open(last_frame_path), dtype=np.uint8, copy=True)).unsqueeze(0)
                    for i in range(num):
                        video_frames.append(first_frame)
                    # add zeros to frames
                    num_zeros = self.num_frames-2*num
                    for i in range(num_zeros):
                        zeros = torch.zeros_like(first_frame)
                        video_frames.append(zeros)
                    for i in range(num):
                        video_frames.append(last_frame)
                    n = 0
                    video_frames = torch.cat(video_frames, dim=0).permute(0, 3, 1, 2) # f,c,h,w
                    video_frames = transform_video(video_frames)
                else:
                    for file in file_list:
                        if file.endswith('jpg') or file.endswith('png'):
                            image = torch.as_tensor(np.array(Image.open(file), dtype=np.uint8, copy=True)).unsqueeze(0)
                            video_frames.append(image)
                        else:
                            continue
                    n = 0
                    video_frames = torch.cat(video_frames, dim=0).permute(0, 3, 1, 2) # f,c,h,w
                    video_frames = transform_video(video_frames)
                return video_frames, n
            elif os.path.isfile(input_path):
                _, full_file_name = os.path.split(input_path)
                file_name, extension = os.path.splitext(full_file_name)
                if extension == '.jpg' or extension == '.png':
                    print("Loading the input image...")
                    video_frames = []
                    num = int(self.mask_type.split('first')[-1])
                    first_frame = torch.as_tensor(np.array(Image.open(input_path).convert('RGB'), dtype=np.uint8, copy=True)).unsqueeze(0)
                    for i in range(num):
                        video_frames.append(first_frame)
                    num_zeros = self.num_frames-num
                    for i in range(num_zeros):
                        zeros = torch.zeros_like(first_frame)
                        video_frames.append(zeros)
                    n = 0
                    video_frames = torch.cat(video_frames, dim=0).permute(0, 3, 1, 2) # f,c,h,w
                    video_frames = transform_video(video_frames)
                    return video_frames, n
                else:
                    raise TypeError(f'{extension} is not supported !!')
            else:
                raise ValueError('Please check your path input!!')
        else:
            raise ValueError('Need to give a video or some images')

    def auto_inpainting(self, video_input, masked_video, mask, prompt, negative_prompt):
        b,f,c,h,w = video_input.shape

        # prepare inputs
        if self.use_fp16:
            z = torch.randn(1, 4, self.num_frames, self.latent_h, self.latent_w, dtype=torch.float16, device=self.device) # b,c,f,h,w
            masked_video = masked_video.to(dtype=torch.float16)
            mask = mask.to(dtype=torch.float16)
        else:
            z = torch.randn(1, 4, self.num_frames, self.latent_h, self.latent_w, device=self.device) # b,c,f,h,w

        masked_video = rearrange(masked_video, 'b f c h w -> (b f) c h w').contiguous()
        masked_video = self.vae.encode(masked_video).latent_dist.sample().mul_(0.18215)
        masked_video = rearrange(masked_video, '(b f) c h w -> b c f h w', b=b).contiguous()
        mask = torch.nn.functional.interpolate(mask[:,:,0,:], size=(self.latent_h, self.latent_w)).unsqueeze(1)
    
        # classifier_free_guidance
        if self.do_classifier_free_guidance:
            masked_video = torch.cat([masked_video] * 2)
            mask = torch.cat([mask] * 2)
            z = torch.cat([z] * 2)
            prompt_all = [prompt] + [negative_prompt]

        else:
            masked_video = masked_video
            mask = mask
            z = z
            prompt_all = [prompt]

        text_prompt = self.text_encoder(text_prompts=prompt_all, train=False)
        model_kwargs = dict(encoder_hidden_states=text_prompt, 
                                class_labels=None, 
                                cfg_scale=self.cfg_scale,
                                use_fp16=self.use_fp16,) # tav unet

        # sample video
        if self.sample_method == 'ddim':
            samples = self.diffusion.ddim_sample_loop(
                self.model.forward_with_cfg, z.shape, z, clip_denoised=False, model_kwargs=model_kwargs, progress=True, device=self.device, \
                mask=mask, x_start=masked_video, use_concat=self.use_mask
            )
        elif self.sample_method == 'ddpm':
            samples = self.diffusion.p_sample_loop(
                self.model.forward_with_cfg, z.shape, z, clip_denoised=False, model_kwargs=model_kwargs, progress=True, device=self.device, \
                mask=mask, x_start=masked_video, use_concat=self.use_mask
            )
        samples, _ = samples.chunk(2, dim=0) # [1, 4, 16, 32, 32]
        if self.use_fp16:
            samples = samples.to(dtype=torch.float16)

        video_clip = samples[0].permute(1, 0, 2, 3).contiguous() # [16, 4, 32, 32]
        video_clip = self.vae.decode(video_clip / 0.18215).sample # [16, 3, 256, 256]

        return video_clip

    def generate_video(self, args, save_path):
        prompt = args.text_prompt
        
        if prompt == []:
            prompt = args.input_path.split('/')[-1].split('.')[0].replace('_', ' ')
        else:
            prompt = prompt[0]
        prompt_base = prompt.replace(' ','_')

        if not os.path.exists(os.path.join(save_path)):
            os.makedirs(os.path.join(save_path))
        video_input, reserve_frames = self.get_input(args.input_path) # f,c,h,w
        video_input = video_input.to(self.device).unsqueeze(0)  # b,f,c,h,w
        mask = mask_generation_before(self.mask_type, video_input.shape, video_input.dtype, self.device) # b,f,c,h,w
        masked_video = video_input * (mask == 0)

        video_clip = self.auto_inpainting(video_input, masked_video, mask, prompt, args.negative_prompt)
        video_ = ((video_clip * 0.5 + 0.5) * 255).add_(0.5).clamp_(0, 255).to(dtype=torch.uint8).cpu().permute(0, 2, 3, 1)
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"{prompt_base}_{timestamp}.mp4"
        save_video_path = os.path.join(save_path, filename)
        save_video_path = os.path.join(save_path, prompt_base + '.mp4')
        torchvision.io.write_video(save_video_path, video_, fps=8)
        print(f'Video saved in {save_video_path}')

        return save_video_path


/home/brina/miniconda3/envs/seine/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load SEINE model with configs

In [2]:
parser = argparse.ArgumentParser()
parser.add_argument("--config", type=str, default="./configs/seine.yaml")
args, unknown = parser.parse_known_args()
omega_conf = OmegaConf.load(args.config)
seine_model = SeineModel(omega_conf)

Initializing SEINE model...
Initialization complete!


## PromptPilot Agent

In [3]:
from openai import OpenAI
import yaml
import re
from argparse import Namespace
import base64
from PIL import Image
from typing import List, Dict
from transformers import (
    CLIPProcessor, CLIPModel
)
import torchvision.transforms as T
import cv2
import numpy as np
import json

# Load CLIP
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

class PromptPilotAgent:
    def __init__(self, seine_model, device="cuda"):
        self.temperature = 0.0
        self.max_iterations = 10
        self.seine_model = seine_model
        self.device = device

        # Load CLIP
        # self.clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
        # self.clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        self.resize = T.Resize((224, 224))
        self.to_tensor = T.ToTensor()
        self.scores = {
            "clip_tva_score": 0,
            "temporal_consistency": 0,
            "dynamic_degree": 0}

    def extract_frames(self,video_path: str, num_frames: int = 4) -> List[Image.Image]:
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_idxs = np.linspace(0, total_frames - 1, num_frames).astype(int)

        frames = []
        for idx in frame_idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(Image.fromarray(rgb_frame))
        cap.release()
        return frames

    def compute_clip_alignment(self, frames: List[Image.Image], prompt: str) -> float:
        inputs = clip_processor(
            text=[prompt] * len(frames), images=frames,
            return_tensors="pt", padding=True
        ).to(self.device)
        with torch.no_grad():
            outputs = clip_model(**inputs)
            sims = torch.cosine_similarity(outputs.image_embeds, outputs.text_embeds)
        return sims.mean().item()
    
    def compute_temporal_consistency(self, frames: List[Image.Image]) -> float:
        gray_frames = [cv2.cvtColor(np.array(f), cv2.COLOR_RGB2GRAY) for f in frames]
        total_flow = 0.0
        for i in range(1, len(gray_frames)):
            flow = cv2.calcOpticalFlowFarneback(
                gray_frames[i - 1], gray_frames[i], None,
                pyr_scale=0.5, levels=3, winsize=15, iterations=3,
                poly_n=5, poly_sigma=1.2, flags=0
            )
            magnitude = np.linalg.norm(flow, axis=2).mean()
            total_flow += magnitude
        return total_flow / (len(frames) - 1)


    def compute_dynamic_degree(self, frames: List[Image.Image]) -> float:
        """
        Computes dynamic degree as the variance of frame-to-frame pixel differences.
        Higher values imply more movement or dynamic content.
        """
        gray_frames = [cv2.cvtColor(np.array(f), cv2.COLOR_RGB2GRAY) for f in frames]
        diffs = []

        for i in range(1, len(gray_frames)):
            diff = np.abs(gray_frames[i].astype(np.float32) - gray_frames[i - 1].astype(np.float32))
            mean_diff = diff.mean()
            diffs.append(mean_diff)

        return float(np.var(diffs)) if diffs else 0.0

    def evaluate_video(self, frames: List[Image.Image], prompt: str) -> Dict[str, float]:
        print("frames",frames)
        clip_score = self.compute_clip_alignment(frames, prompt)
        tc_score = self.compute_temporal_consistency(frames)
        dd_score = self.compute_dynamic_degree(frames)

        self.scores.update({
            "clip_tva_score": clip_score,
            "temporal_consistency": tc_score,
            "dynamic_degree": dd_score
        })

        return self.scores  # Return updated dictionary if needed
    
    def set_first_text_prompt(self, yaml_path):
        with open(yaml_path, 'r') as f:
            content = yaml.safe_load(f)

        # Extract the input_path value to use as the new_prompt
        input_path = content.get("input_path", "No input_path found")
        prompt = input_path.split('/')[-1].split('.')[0].replace('_', ' ')
    
        with open(yaml_path, 'r') as f:
            content = f.read()

        # replace the text_prompt line
        new_content = re.sub(
            r'text_prompt:.*?\n',
            f'text_prompt: [{prompt}]\n',
            content
        )

        with open(yaml_path, 'w') as f:
            f.write(new_content)

    def update_text_prompt(self, yaml_path, refined_prompt):
        with open(yaml_path, 'r') as f:
            content = f.read()

        # replace the text_prompt line
        new_content = re.sub(
            r'text_prompt:.*?\n',
            f'text_prompt: [{refined_prompt}]\n',
            content
        )

        with open(yaml_path, 'w') as f:
            f.write(new_content)
    
    def cleanup(self):
        torch.cuda.empty_cache()

    def run(self, yaml_path):
        print("Generating video...")

        # set text prompt for the first time
        self.set_first_text_prompt(yaml_path)

        # Read the file content
        with open(yaml_path, 'r') as f:
            content = yaml.safe_load(f)

        # get the string inside the text_prompt list
        prompt = content.get("text_prompt", [None])[0]
        save_path = content.get("save_path", None)
        save_path = save_path.replace("./results/", "./results/exp1/")

        # prepare args for video generation
        with open(yaml_path, "r") as f:
            content = yaml.safe_load(f)
        args = Namespace(**content)
        video_path = self.seine_model.generate_video(args, save_path)

        # compute evaluation metrics
        frames = self.extract_frames(video_path)
        scores = self.evaluate_video(frames, prompt)
        print(f"Prompt: {prompt}")
        print(f"Scores: {scores}")

        # Save to JSON
        results = {
            "prompt": prompt,
            "scores": {k: float(v) for k, v in scores.items()}  # Ensure all values are JSON-serializable
        }
        with open(f"{save_path}/scores.json", "w") as f:
            json.dump(results, f, indent=4)

        # cleanup after generation
        self.cleanup()

## Create Prompt Agent

In [4]:
agent = PromptPilotAgent(seine_model)

directory = "configs/images"

# Using os.walk to loop through subdirectories
yaml_paths = []
for subdir, _, files in os.walk(directory):
    for file in files:
        if file.endswith('.yaml'):
            # Add the full path of each YAML file
            yaml_paths.append(os.path.join(subdir, file))

# Process each YAML file
for yaml_path in yaml_paths:
    print(f"\nProcessing: {yaml_path}")
    agent.run(yaml_path)


Processing: configs/images/Animals/cats_looking_around.yaml
Generating video...
Loading video from dataset/Animals/cats_looking_around.png...
Loading the input image...


100%|██████████| 250/250 [03:20<00:00,  1.25it/s]


Video saved in ./results/exp1/Animals/cats_looking_around/cats_looking_around_20250425_104908.mp4
frames [<PIL.Image.Image image mode=RGB size=480x320 at 0x73A1236FC2E0>, <PIL.Image.Image image mode=RGB size=480x320 at 0x73A1236FC280>, <PIL.Image.Image image mode=RGB size=480x320 at 0x73A1236FC220>, <PIL.Image.Image image mode=RGB size=480x320 at 0x73A1236E3D60>]
Prompt: cats looking around
Scores: {'clip_tva_score': 0.2662765681743622, 'temporal_consistency': 0.5763820012410482, 'dynamic_degree': 0.011112394742667675}

Processing: configs/images/Animals/white_tiger_walking.yaml
Generating video...
Loading video from dataset/Animals/white_tiger_walking.png...
Loading the input image...


100%|██████████| 250/250 [03:24<00:00,  1.23it/s]


Video saved in ./results/exp1/Animals/white_tiger_walking/white_tiger_walking_20250425_105235.mp4
frames [<PIL.Image.Image image mode=RGB size=480x320 at 0x73A1FDFF61F0>, <PIL.Image.Image image mode=RGB size=480x320 at 0x73A1FDFF6220>, <PIL.Image.Image image mode=RGB size=480x320 at 0x73A123541FA0>, <PIL.Image.Image image mode=RGB size=480x320 at 0x73A123542FD0>]
Prompt: white tiger walking
Scores: {'clip_tva_score': 0.3116731345653534, 'temporal_consistency': 4.02263347307841, 'dynamic_degree': 1.8646255731582642}

Processing: configs/images/Animals/close_up_giraffe.yaml
Generating video...
Loading video from dataset/Animals/close_up_giraffe.png...
Loading the input image...


...

Video saved in ./results/exp1/Outdoor/waterfall_and_moving_streams/waterfall_and_moving_streams_20250426_022521.mp4
frames [<PIL.Image.Image image mode=RGB size=480x320 at 0x73A12354A880>, <PIL.Image.Image image mode=RGB size=480x320 at 0x73A12354AEB0>, <PIL.Image.Image image mode=RGB size=480x320 at 0x73A12354ABE0>, <PIL.Image.Image image mode=RGB size=480x320 at 0x73A12354A7F0>]
Prompt: waterfall and moving streams
Scores: {'clip_tva_score': 0.2596743106842041, 'temporal_consistency': 0.7199635704358419, 'dynamic_degree': 0.14705263078212738}

Processing: configs/images/Outdoor/metropolitan_evening_view.yaml
Generating video...
Loading video from dataset/Outdoor/metropolitan_evening_view.png...
Loading the input image...


100%|██████████| 250/250 [03:18<00:00,  1.26it/s]


Video saved in ./results/exp1/Outdoor/metropolitan_evening_view/metropolitan_evening_view_20250426_022844.mp4
frames [<PIL.Image.Image image mode=RGB size=480x320 at 0x73A123542FD0>, <PIL.Image.Image image mode=RGB size=480x320 at 0x73A1FDFF61F0>, <PIL.Image.Image image mode=RGB size=480x320 at 0x73A1FDFF6220>, <PIL.Image.Image image mode=RGB size=480x320 at 0x73A12354AB20>]
Prompt: metropolitan evening view
Scores: {'clip_tva_score': 0.2677781581878662, 'temporal_consistency': 5.089921315511067, 'dynamic_degree': 0.9504449367523193}

Processing: configs/images/Outdoor/huge_vessel_passing_ravine.yaml
Generating video...
Loading video from dataset/Outdoor/huge_vessel_passing_ravine.png...
Loading the input image...


100%|██████████| 250/250 [03:19<00:00,  1.26it/s]


Video saved in ./results/exp1/Outdoor/huge_vessel_passing_ravine/huge_vessel_passing_ravine_20250426_023206.mp4
frames [<PIL.Image.Image image mode=RGB size=480x320 at 0x73A1FE85CAF0>, <PIL.Image.Image image mode=RGB size=480x320 at 0x73A1FE85C6D0>, <PIL.Image.Image image mode=RGB size=480x320 at 0x73A1FE85C640>, <PIL.Image.Image image mode=RGB size=480x320 at 0x73A1FE85CB20>]
Prompt: huge vessel passing ravine
Scores: {'clip_tva_score': 0.25561919808387756, 'temporal_consistency': 6.431050777435303, 'dynamic_degree': 2.8044979572296143}
